# MCDI500 — Fase 1: definición y entorno reproducible
**Proyecto:** Distribución y concentración de la matrícula de primer año de Institutos Profesionales en Chile (2025).  
**Grupo 8 — integrantes confirmados:** Cristian Pérez Cortés y Claudio Navarrete Sims.  
**Docente:** Omar Iván Salinas Silva. **Fuente:** Libro INDICES CNED aportado por el equipo.  
**Delimitación:** F1 *define y reconoce*; F2 *obtiene, limpia, transforma y valida*. La Fase 1 **no imputa ni transforma variables**.


## 1. Pregunta, objetivos, alcance, restricciones y supuestos
**Pregunta:** ¿Cómo se distribuye y concentra la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, según institución, región y área de conocimiento?

**Objetivo general:** Analizar la distribución y la concentración de la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, por institución, región y área de conocimiento, mediante un flujo de procesamiento y análisis descriptivo reproducible en Python.

**Objetivos específicos analíticos (F1–F3):**
1. Describir la distribución de la matrícula de primer año de los Institutos Profesionales durante 2025 por institución, región y área de conocimiento, cuantificando totales y participaciones.
2. Medir y comparar la concentración de dicha matrícula entre instituciones, regiones y áreas mediante los indicadores CR3 y HHI.
3. Evaluar la sensibilidad del total de matrícula de primer año ante la anomalía temporal identificada en el registro de IP Santo Tomás, sin atribuir relaciones causales.

**Actividades metodológicas (no objetivos de investigación):** seleccionar y perfilar registros en F1; comparar imputaciones, documentar transformaciones y validar datos en F2; verificar la arquitectura algorítmica y su eficiencia en F3.

**Alcance decidido:** registros IP año 2025; pregrado y posgrado se presentan juntos para conservar la comparación con la entrega original (4.714 y 17, respectivamente). Análisis descriptivo de matrícula agregada de primer año. **Excluye:** causalidad, deserción, predicción individual y variaciones longitudinales.

**Restricciones:** fuente secundaria con faltantes y algunas inconsistencias; archivo mayor a 60 MB; periodo académico de tres semanas; URL de la versión exacta 2005–2026 pendiente de identificación; repositorio GitHub ajeno al entorno de ejecución.

**Supuestos:** cada fila representa una observación agregada de oferta académica (no un estudiante); los ceros de matrícula se mantienen; no se infiere información ausente en identificadores; la fuente suministrada se conserva íntegra y se identifica con SHA-256.

**Criterios de éxito:** ejecución de F1 y F2 sin errores, 4.731 filas, totales 137.481 y 439.607, integridad de observaciones originales, decisiones justificadas cuantitativamente, y evidencias en `docs/` y `evidencias/`.


In [1]:
PROYECTO = {
    "pregunta": "¿Cómo se distribuye y concentra la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, según institución, región y área de conocimiento?",
    "objetivo": "Analizar la distribución y la concentración de la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, por institución, región y área de conocimiento, mediante un flujo de procesamiento y análisis descriptivo reproducible en Python.",
    "anio": 2025, "tipo": "I.P.", "semilla": 42,
    "fuente": "CNED/INDICES, libro suministrado 2005–2026",
    "unidad": "registro agregado de oferta académica, no persona",
}
print("Pregunta de investigación:", PROYECTO["pregunta"])
print("Semilla:", PROYECTO["semilla"])


Pregunta de investigación: ¿Cómo se distribuye y concentra la matrícula de primer año de los Institutos Profesionales en Chile durante 2025, según institución, región y área de conocimiento?
Semilla: 42


## 2. Dependencias, intérprete y estructura de directorios
Esta sección **verifica** las librerías y el intérprete que está ejecutando el notebook; **no instala paquetes durante la ejecución**. Los paquetes se instalan previamente usando el archivo `requirements.txt` y un entorno `.venv`. Cada notebook se puede abrir desde su carpeta de fase o desde la raíz del repositorio.

In [2]:
from pathlib import Path
import json, sys, platform, importlib, subprocess
from importlib import metadata as imeta
import pandas as pd
import numpy as np
from IPython.display import display
np.random.seed(42)
RAIZ=Path.cwd().resolve()
if RAIZ.name in {"F1","F2","F3","F4"}: RAIZ=RAIZ.parent
if not (RAIZ/"src/cned_pipeline.py").exists():
    raise FileNotFoundError("Ejecutar desde la raíz del proyecto o su carpeta F1")
if str(RAIZ) not in sys.path: sys.path.insert(0,str(RAIZ))
from src.cned_pipeline import cargar_cned_ip, validar_fase1, diagnosticar_faltantes, promedio_matricula, sha256_archivo
RUTA_EXCEL=RAIZ/"data/raw/Base-INDICES-Matricula-2005-2026.xlsx"
RUTA_FUENTE=RUTA_EXCEL if RUTA_EXCEL.is_file() else RAIZ/"data/fixtures/ip_2025_origen_filtrado.csv"
DIR_DOCS=RAIZ/"docs"; DIR_DOCS.mkdir(parents=True,exist_ok=True)
paquetes=["pandas","numpy","openpyxl","lxml","matplotlib","jupyterlab","ipykernel","nbformat","nbclient","pytest"]
print("Python",sys.version.split()[0],"| intérprete:",sys.executable)
print("Virtualenv:",sys.prefix!=sys.base_prefix,"| Sistema:",platform.system())
versiones={}
for p in paquetes:
    v=imeta.version(p); versiones[p]=v; print(f"[OK] {p:12} {v}")
assert RUTA_FUENTE.is_file(), f"Falta el Excel CNED y el CSV filtrado publicado: {RUTA_FUENTE}"
assert (RAIZ/"requirements.txt").exists()
print("Fuente de ejecución:",RUTA_FUENTE.relative_to(RAIZ))
print("SHA-256 fuente:",sha256_archivo(RUTA_FUENTE))


Python 3.13.5 | intérprete: /opt/pyvenv/bin/python
Virtualenv: True | Sistema: Linux
[OK] pandas       2.2.3
[OK] numpy        2.3.5
[OK] openpyxl     3.1.5
[OK] lxml         6.1.1
[OK] matplotlib   3.10.8
[OK] jupyterlab   4.5.3
[OK] ipykernel    7.2.0
[OK] nbformat     5.10.4
[OK] nbclient     0.10.4
[OK] pytest       9.0.2
SHA-256 fuente: 0439bc1ed1a6ec018a3bbf4be34861bd33749c81c34c2eeeddbc7dcd2657c48a


## 3. Obtención y selección reproducible de registros
El libro Excel tiene más de 260 mil filas. La función reutilizable `cargar_cned_ip()` extrae, mediante lectura incremental del XML XLSX, **solo** `Año=2025` y `Tipo Institución=I.P.`, manteniendo intactas las 62 columnas y el Excel original. No se introduce información artificial.

In [3]:
datos_ip, procedencia = cargar_cned_ip(RUTA_FUENTE)
display(pd.DataFrame([procedencia]))
display(datos_ip[["Nombre Institución","Nombre Region","Area Conocimiento", "Matrícula Primer Año", "Matrícula Total"]].head())
assert procedencia["filas_origen"] == (4731 if procedencia.get("fuente_filtrada") else 260307)
assert procedencia["columnas_origen"] == 62
assert datos_ip.shape == (4731,62)

,archivo_origen,sha256,filas_origen,columnas_origen,filas_filtradas,anio,tipo_institucion
0,Base-INDICES-Matricula-2005-2026.xlsx,0439bc1ed1a6ec018a3bbf4be34861bd33749c81c34c2e...,260307,62,4731,2025,I.P.


,Nombre Institución,Nombre Region,Area Conocimiento,Matrícula Primer Año,Matrícula Total
0,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",72,367
1,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",41,144
2,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",69,166
3,IP AGRARIO ADOLFO MATTHEI,Región de Los Lagos,"Agricultura, Silvicultura, Pesca y Veterinaria",41,79
4,IP INACAP,Región de Antofagasta,Administración y Comercio,29,165


## 4. Perfil de las variables y calidad inicial
Una cifra cero de matrícula significa «cero observado» y no «valor ausente». Los códigos de sede/campus son identificadores, aun cuando Python los almacene como números. Antes de transformar, se examinan: tipos, ausencias, duplicados, componentes de matrícula y la anomalía del registro cuya fecha de inicio es posterior a 2025.

In [4]:
validacion_f1 = validar_fase1(datos_ip)
print(json.dumps(validacion_f1,ensure_ascii=False,indent=2))
print("Promedio observado de matrícula total por registro:",round(promedio_matricula(datos_ip),2))
print("Distribución pregrado/posgrado:")
display(datos_ip["Pregrado/Posgrado"].value_counts(dropna=False).to_frame("registros"))
display(diagnosticar_faltantes(datos_ip).head(18))

{
  "registros": 4731,
  "instituciones": 27,
  "columnas": 62,
  "matricula_primer_anio": 137481,
  "matricula_total": 439607,
  "registro_inicio_posterior": 1,
  "duplicados_completos": 0,
  "matricula_primer_anio_cero": 1576,
  "pregrado": 4714,
  "posgrado": 17
}
Promedio observado de matrícula total por registro: 92.92
Distribución pregrado/posgrado:


,registros
Pregrado/Posgrado,
Pregrado,4714
Posgrado,17


,variable,faltantes,porcentaje_faltante,tipo
0,Máximo Puntaje (promedio matemáticas y lenguaje),4731,100.000,string
1,Máximo Puntaje NEM,4731,100.000,string
2,Máximo Puntaje Ranking,4731,100.000,string
3,Mínimo Puntaje (promedio matemáticas y lenguaje),4731,100.000,string
4,Mínimo Puntaje NEM,4731,100.000,string
5,Mínimo Puntaje Ranking,4731,100.000,string
6,Nº Alumnos Ingreso Via PSU o PDT,4731,100.000,float64
7,Promedio Puntaje (promedio matemáticas y lengu...,4731,100.000,string
8,Promedio Puntaje NEM,4731,100.000,string
9,Promedio Puntaje Ranking,4731,100.000,string


### 4.1 Anomalías temporales y posible duplicación
Antes de decidir si un registro es incorrecto se muestran los casos de fecha posterior a 2025 y las coincidencias por clave. Una coincidencia de institución, programa y sede no prueba duplicación: puede diferir campus, horario o modalidad. No se elimina ningún registro en F1.

In [5]:
print("Registros con inicio de actividades posterior a 2025:")
anomalias=datos_ip.loc[datos_ip["Año Inicio Actividades"]>2025,
  ["Nombre Institución","Nombre Programa","Año","Año Inicio Actividades","Matrícula Primer Año","Matrícula Total"]]
display(anomalias)
print("Duplicados exactos:",int(datos_ip.duplicated().sum()))
claves=["Nombre Institución","Nombre Programa","Nombre de la Sede"]
print("Coincidencias institución/programa/sede (no necesariamente duplicados):",
      int(datos_ip.duplicated(subset=claves,keep=False).sum()))
assert validacion_f1["registro_inicio_posterior"]==1


Registros con inicio de actividades posterior a 2025:


,Nombre Institución,Nombre Programa,Año,Año Inicio Actividades,Matrícula Primer Año,Matrícula Total
4716,IP SANTO TOMÁS,Diplomado en Liderazgo y Gestión de Personas,2025,2026,39,39


Duplicados exactos: 0
Coincidencias institución/programa/sede (no necesariamente duplicados): 3171


## 5. Diccionario de datos y roles analíticos
La clasificación se realiza por **semántica**, no únicamente mediante `dtype`: matrícula y vacantes son conteos discretos, aranceles son valores monetarios, códigos son identificadores, región/institución/área son variables nominales y los años son temporales. Conservar el diccionario permite justificar transformaciones en F2.

In [6]:
def definir_rol(columna):
    nombre=columna.casefold()
    if nombre.startswith("cód") or nombre.startswith("id") or nombre=="códgo sies":
        return "identificador"
    if "matrícula" in nombre and "valor" not in nombre:
        return "conteo discreto"
    if "arancel" in nombre or "valor" in nombre:
        return "monetaria"
    if "año" in nombre:
        return "temporal"
    if nombre in {"vacantes","duración (en semestres)","nº alumnos ingreso via psu o pdt","nº alumnos ingreso otra via"}:
        return "conteo discreto"
    return "nominal/categórica"

diccionario = pd.DataFrame({
    "variable":datos_ip.columns,
    "tipo_python":datos_ip.dtypes.astype(str).values,
    "rol_analitico":[definir_rol(c) for c in datos_ip.columns],
    "faltantes":datos_ip.isna().sum().values,
    "valores_distintos":[datos_ip[c].nunique(dropna=True) for c in datos_ip.columns],
})
diccionario.to_csv(DIR_DOCS/"diccionario_variables.csv",index=False,encoding="utf-8-sig")
display(diccionario.head(12))
assert len(diccionario)==62

,variable,tipo_python,rol_analitico,faltantes,valores_distintos
0,Año,int64,temporal,0,1
1,Cód. Institución,object,identificador,0,27
2,Nombre Institución,object,nominal/categórica,0,27
3,Tipo Institución,object,nominal/categórica,0,1
4,Clasificación1,object,nominal/categórica,0,1
5,Clasificación2,object,nominal/categórica,0,1
6,Clasificación3,object,nominal/categórica,0,2
7,Clasificación4,object,nominal/categórica,0,2
8,Clasificación5,object,nominal/categórica,0,2
9,Clasificación6,object,nominal/categórica,0,2


## 6. Validación del dataset conforme a la guía del docente
La plantilla docente `validar_dataset.py` pide al menos 2.000 registros, 12 columnas, tres roles, variables numéricas y categóricas. Nuestro subconjunto presenta 4.731 registros y 62 columnas; se evalúa por unidad **filtrada** (IP/2025). Advertencia: la ausencia del 100% en algunas variables no invalida la matrícula, pero exige documentar su descarte y no considerarlas predictores.

In [7]:
criterios = pd.DataFrame([
    {"criterio":"Registros ≥ 2.000","valor":len(datos_ip),"cumple":len(datos_ip)>=2000},
    {"criterio":"Columnas ≥ 12","valor":datos_ip.shape[1],"cumple":datos_ip.shape[1]>=12},
    {"criterio":"Roles analíticos ≥ 3","valor":diccionario["rol_analitico"].nunique(),"cumple":diccionario["rol_analitico"].nunique()>=3},
    {"criterio":"Numéricas presentes","valor":len(datos_ip.select_dtypes(include='number').columns),"cumple":len(datos_ip.select_dtypes(include='number').columns)>0},
    {"criterio":"Categóricas presentes","valor":len(datos_ip.select_dtypes(exclude='number').columns),"cumple":len(datos_ip.select_dtypes(exclude='number').columns)>0},
])
display(criterios)
assert criterios["cumple"].all()
criterios.to_csv(DIR_DOCS/"evaluacion_criterios_dataset.csv",index=False,encoding="utf-8-sig")

,criterio,valor,cumple
0,Registros ≥ 2.000,4731,True
1,Columnas ≥ 12,62,True
2,Roles analíticos ≥ 3,5,True
3,Numéricas presentes,18,True
4,Categóricas presentes,44,True


## 7. Registro de metadatos, Git y trazabilidad (sin transformar datos)
La Fase 1 conserva el Excel sin alteraciones. Registra la fuente, la identidad de su intérprete, las versiones de cada dependencia y, cuando existe un repositorio local, la rama/commit. **No genera un CSV derivado**: la obtención y persistencia de datos son responsabilidad de F2.


In [8]:
def git_lectura(*args):
    try:
        proceso=subprocess.run(["git",*args],cwd=RAIZ,capture_output=True,text=True,timeout=5)
        return proceso.stdout.strip() if proceso.returncode==0 else "No disponible (copia sin .git)"
    except (OSError,subprocess.TimeoutExpired):
        return "No disponible (Git no instalado)"
metadata={
    "fase":"F1","pregunta":PROYECTO["pregunta"],"objetivo":PROYECTO["objetivo"],
    "fuente":"CNED INDICES: libro 2005–2026 suministrado",
    "url_portal":"https://cned.cl/institucional/bases-de-datos/",
    "url_descarga_2005_2026":"no acreditada",
    "sha256_origen":procedencia.get("sha256_excel_que_lo_origino",procedencia["sha256"]),
    "fuente_ejecutada":str(RUTA_FUENTE.relative_to(RAIZ)),
    "sha256_fuente_ejecutada":procedencia["sha256"],
    "fuente_csv_filtrada":bool(procedencia.get("fuente_filtrada")),
    "filas_fuente_ejecutada":procedencia["filas_origen"],
    "registros_origen":260307 if procedencia.get("fuente_filtrada") else procedencia["filas_origen"],
    "registros_ip_2025":len(datos_ip),"variables":datos_ip.shape[1],
    "matricula_primer_anio":validacion_f1["matricula_primer_anio"],
    "matricula_total":validacion_f1["matricula_total"],
    "python":sys.version.split()[0],"intérprete":sys.executable,
    "entorno_virtual":sys.prefix!=sys.base_prefix,"dependencias":versiones,
    "git_rama":git_lectura("rev-parse","--abbrev-ref","HEAD"),
    "git_commit":git_lectura("rev-parse","HEAD"),
}
(DIR_DOCS/"metadatos_fase1.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Documento de procedencia:",DIR_DOCS/"metadatos_fase1.json")
print("Git rama:",metadata["git_rama"],"| commit:",metadata["git_commit"])
assert not (RAIZ/"data/raw"/"README.md").samefile(RUTA_FUENTE)


Documento de procedencia: /mnt/data/CNED_MCDI500_Sumativa1_Guia_Integrada/docs/metadatos_fase1.json
Git rama: No disponible (copia sin .git) | commit: No disponible (copia sin .git)


## 8. Vinculación con el mapa conceptual y fases siguientes
| Nodo conceptual | Implementación F1 | Evidencia |
|---|---|---|
| Problema y alcance | Definición única de pregunta, objetivos, exclusiones | Sección 1 y `PROYECTO` |
| Fuente y trazabilidad | Libro sin modificar, hash SHA-256 y filtrado 2025/IP | Secciones 3 y 7 |
| Ecosistema y entorno | Python, Pandas, NumPy, LXML, Jupyter | Sección 2, `requirements.txt` |
| Validación preliminar | Calidad, duplicados, anomalía temporal | Secciones 4–6 |
| Limpieza y variables derivadas | F2 | `F2/F2_Preprocesamiento_Cned.ipynb` |
| Algoritmos / POO | F3 | `F3/S2_F3_NucleoAlgoritmico_Cned.ipynb` (implementado después de F2) |
| Visualizaciones de resultados | F4 | `F4/` (proyectado) |

**Decisión temporal:** se conserva el programa de IP Santo Tomás cuyo año de inicio registrado es 2026 y se etiqueta como inconsistencia; se evaluará sensibilidad sin alterar la base.

## 9. Conclusión de F1
Se delimita una pregunta de distribución y concentración de matrícula de primer año (no una predicción). El dataset de IP/2025 es válido para análisis descriptivo; la exactitud de su URL de descarga 2005–2026 todavía debe documentarse. La Fase 2 conservará la matrícula original, tratará los faltantes según el rol de cada variable, comparará estrategias de imputación y verificará que no cambia ninguna cifra observada.